# European vs US Private Credit: Portfolio Construction Using Public Proxies

**Objective:** Compare a US high-yield credit proxy with a European high-yield credit proxy, quantify risk/return and tail behavior, and evaluate simple static blends as a *portfolio construction* exercise.

**Audience framing:** This notebook is written in the style of a short internal research note—clear exhibits, minimal machinery, explicit assumptions.


## Why public proxies instead of private loan-level data?

Private credit portfolios are **not** trivial to study with fully public, mark-to-market time series. Loan-level cashflows, covenants, and manager-specific sourcing are largely **non-public**, and fund returns can reflect **valuation smoothing** and **liquidity terms** that ETFs do not replicate.

To keep the project **honest and reproducible**, we use **listed high-yield corporate bond ETFs** as **transparent proxies** for *regional credit risk*:

- **USHY** — broad **USD** high-yield corporate exposure (US proxy).
- **IHYG** — **EUR** high-yield corporate exposure (European proxy).

These are **public market** instruments. Any statement about “private credit” here is **economic intuition** about regional credit beta sleeves—not a claim that ETF performance equals private fund performance.


In [10]:
# If you launch Jupyter from `notebooks/`, add repo root + src to import path
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import warnings

import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf

from utils import (
    cumulative_returns,
    max_drawdown,
    portfolio_returns,
    summary_statistics_table,
    compute_drawdown_series,
)

warnings.filterwarnings("ignore", category=FutureWarning)

for _style in ("seaborn-v0_8-whitegrid", "seaborn-whitegrid"):
    try:
        plt.style.use(_style)
        break
    except OSError:
        continue

plt.rcParams.update(
    {
        "figure.figsize": (10, 5),
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "legend.fontsize": 10,
        "font.size": 11,
    }
)

FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")


ModuleNotFoundError: No module named 'matplotlib'

## Configuration: tickers, sample window, and fallbacks

We download **adjusted** prices so splits and distributions do not distort return series. If a primary ticker fails (symbol changes, delisting, or API quirks), we fall back to sensible alternatives and print the resolved mapping.

**Fallback policy (document any change in your README):**

- US: `USHY` → `HYG` → `JNK`
- Europe: `IHYG` → `IHYG.L` (London listing) → `SHYU.L` (EUR HY UCITS; verify on Yahoo if needed)


In [ ]:
START = "2015-01-01"
END = None  # None = latest available from Yahoo Finance

FALLBACKS = {
    "us_hy": ["USHY", "HYG", "JNK"],
    "eu_hy": ["IHYG", "IHYG.L", "SHYU.L"],
}


def download_close(ticker: str, start: str, end: str | None) -> pd.Series:
    df = yf.download(
        ticker,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False,
        threads=False,
    )
    if df.empty:
        return pd.Series(dtype=float)

    col = "Close" if "Close" in df.columns else df.columns[0]
    s = df[col].copy()
    s.name = ticker
    s.index = pd.to_datetime(s.index).tz_localize(None)
    return s.sort_index()


def load_first_available(
    keys_order: list[str],
    fallbacks_map: dict[str, list[str]],
    start: str,
    end: str | None,
) -> tuple[pd.DataFrame, dict[str, str]]:
    closes: dict[str, pd.Series] = {}
    resolved: dict[str, str] = {}

    for logical in keys_order:
        chosen = None
        for t in fallbacks_map[logical]:
            s = download_close(t, start=start, end=end)
            if not s.empty:
                chosen = t
                closes[logical] = s
                break
        if chosen is None:
            raise RuntimeError(f"Could not download any ticker for {logical}: {fallbacks_map[logical]}")
        resolved[logical] = chosen

    panel = pd.concat(closes, axis=1)
    panel = panel.sort_index().dropna(how="any")
    return panel, resolved


prices, resolved_tickers = load_first_available(
    keys_order=["us_hy", "eu_hy"],
    fallbacks_map=FALLBACKS,
    start=START,
    end=END,
)

print("Resolved tickers:")
for k, v in resolved_tickers.items():
    print(f"  {k}: {v}")

prices.tail()


## Clean and align the time series

We require overlapping dates for both proxies. Missing days are dropped **jointly** so returns are comparable.


In [ ]:
prices = prices.dropna(how="any")

assert prices.shape[0] > 250, "Not enough overlapping observations—check tickers/date range."

prices.head(), prices.tail()


## Daily simple returns

We use **simple** daily returns: \(r_t = P_t / P_{t-1} - 1\) for readability in portfolio aggregation.


In [ ]:
returns = prices.pct_change().dropna()
returns = returns.rename(columns={"us_hy": "US HY proxy", "eu_hy": "Europe HY proxy"})

returns.head()


## Exhibit: normalized price levels (rebased to 100)

This is a **total-return level** view after Yahoo’s adjustment. It is useful for intuition, but portfolio statistics below are return-based.


In [ ]:
rebased = 100 * prices / prices.iloc[0]

ax = rebased.plot(linewidth=2)
ax.set_title("Regional HY proxies — rebased prices")
ax.set_ylabel("Index (start = 100)")
ax.legend(["US HY proxy", "Europe HY proxy"])
plt.tight_layout()
plt.savefig(FIG_DIR / "normalized_prices.png", dpi=160)
plt.show()


## Summary statistics (annualized; Sharpe uses rf = 0)

Annualization assumes **252** trading days. Sharpe uses a **0%** risk-free rate as a deliberate simplification—institutional work would use bills/OIS.

**Interpretation guardrail:** higher return often comes with higher drawdown risk in credit; a single Sharpe number is never sufficient for allocator decisions.


In [ ]:
proxy_stats = summary_statistics_table(returns, rf_annual=0.0)
proxy_stats.round(4)


## Correlation analysis

Correlation answers a narrow question: **how much independent motion** exists between sleeves *on this sample*. Positive but imperfect correlation is the usual precondition for blending to matter.


In [ ]:
corr = returns.corr()
corr.round(3)


## Drawdown analysis

Drawdowns emphasize **path risk**—often more salient in allocator conversations than a single volatility number.


In [ ]:
cum_us = cumulative_returns(returns["US HY proxy"])
cum_eu = cumulative_returns(returns["Europe HY proxy"])

dd_us = compute_drawdown_series(cum_us)
dd_eu = compute_drawdown_series(cum_eu)

fig, ax = plt.subplots()
ax.plot(dd_us.index, dd_us, label="US HY proxy")
ax.plot(dd_eu.index, dd_eu, label="Europe HY proxy")
ax.set_title("Drawdown series (wealth / trailing peak - 1)")
ax.set_ylabel("Drawdown")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "drawdowns.png", dpi=160)
plt.show()

print(
    "Max drawdown (US, Europe):",
    round(max_drawdown(returns["US HY proxy"]), 4),
    round(max_drawdown(returns["Europe HY proxy"]), 4),
)


## Static portfolios (long-only, buy-and-hold weights on daily returns)

We compare:

- **100% US** / **100% Europe**
- **50 / 50**
- **70 / 30** and **30 / 70** (US weight first)

Weights are fixed through time (no rebalancing frictions modeled).


In [ ]:
asset_cols = ["US HY proxy", "Europe HY proxy"]
R = returns[asset_cols].copy()

weights = {
    "100% US": {"US HY proxy": 1.0, "Europe HY proxy": 0.0},
    "100% Europe": {"US HY proxy": 0.0, "Europe HY proxy": 1.0},
    "50/50": {"US HY proxy": 0.5, "Europe HY proxy": 0.5},
    "70/30 US/Europe": {"US HY proxy": 0.7, "Europe HY proxy": 0.3},
    "30/70 US/Europe": {"US HY proxy": 0.3, "Europe HY proxy": 0.7},
}

portfolios = pd.DataFrame({name: portfolio_returns(R, w) for name, w in weights.items()})
portfolios.tail()


## Compare portfolio metrics

Same statistics as the proxies, now applied to blended sleeves.


In [ ]:
portfolio_stats = summary_statistics_table(portfolios, rf_annual=0.0)
portfolio_stats.round(4)


## Exhibit: cumulative returns of portfolios

This chart is the cleanest “allocator view” of how static mixes evolved **ex post** over the sample.


In [ ]:
cum_ports = pd.DataFrame({c: cumulative_returns(portfolios[c]) for c in portfolios.columns})

ax = cum_ports.plot(linewidth=2)
ax.set_title("Cumulative simple returns — static portfolios")
ax.set_ylabel("Cumulative return")
plt.tight_layout()
plt.savefig(FIG_DIR / "portfolio_cumulative_returns.png", dpi=160)
plt.show()


## Conclusion (portfolio research tone)

Over the downloaded sample, the two regional **high-yield credit proxies** generally co-move (credit is a global risk factor), but they are **not perfectly correlated**. That leaves room—*in public proxy space*—for **diversification mechanics** to show up when you blend sleeves: interior portfolios often land between the extremes on realized metrics, though **crisis periods** can still synchronize markets and temporarily collapse diversification benefits.

**Honesty statement:** this exercise illustrates a **portfolio construction framework** (weights → portfolio returns → risk metrics) using **listed credit beta**. It should **not** be read as evidence about **private credit fund** behavior, liquidity, or manager skill.

If your recruiting conversation turns to private markets, the defensible takeaway is methodological: you can **frame** a regional sleeve question, **quantify** co-movement and tail risk on transparent data, and **communicate** what is—and is not—being modeled.


## Limitations (read carefully)

- **ETFs ≠ private credit:** differences in sourcing, structures, covenants, fees, reporting, and smoothing make direct performance mapping inappropriate.
- **FX:** a US allocator holding EUR-denominated risk may need **hedging** analysis not captured here.
- **rf = 0:** Sharpe is a simplified student baseline; swap/T-bill funding would change levels (usually not the correlation story).
- **Static weights:** no turnover, transaction costs, or mandate constraints.
- **Vendor noise:** Yahoo data can be revised; snapshot CSVs if you need auditability.

**Next step if you want this on a CV:** run the notebook end-to-end, then paste your actual summary table outputs into the README “Key findings” section (or attach exported figures from `figures/`).
